In [1]:
# ...existing code...
import pandas as pd
import csv
from pathlib import Path
from IPython.display import display

def read_csv_with_fallback(path: Path, delimiter=None):
    # detecta delimitador com csv.Sniffer se não informado
    if delimiter is None:
        try:
            with open(path, "r", encoding="utf-8", errors="replace") as f:
                sample = f.read(8192)
                delim = csv.Sniffer().sniff(sample).delimiter
        except Exception:
            delim = ","
    else:
        delim = delimiter

    # tenta utf-8, depois latin1
    try:
        return pd.read_csv(path, sep=delim, engine="python", encoding="utf-8")
    except Exception:
        return pd.read_csv(path, sep=delim, engine="python", encoding="latin1")

def convert_folder(input_dir: str, output_dir: str = "xlsx_output", recurse: bool = False, delimiter: str = None):
    input_path = Path(input_dir)
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    pattern = "**/*.csv" if recurse else "*.csv"
    files = list(input_path.glob(pattern))
    if not files:
        print("Nenhum .csv encontrado em:", input_path)
        return []

    results = []
    for f in files:
        try:
            print("Lendo:", f.name)
            df = read_csv_with_fallback(f, delimiter)
            out_file = out_path / (f.stem + ".xlsx")
            with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
                df.to_excel(writer, index=False, sheet_name="Sheet1")
            print("Salvo:", out_file)
            results.append(str(out_file))
        except Exception as ex:
            print(f"Erro ao processar {f.name}: {ex}")
    return results

# Exemplo de uso no notebook (ajuste caminhos se quiser)
input_folder = r"C:\Users\gabri\Desktop\FIAP\Fase 4\Aulas\Ligacao_com_banco_de_dados\data"
output_folder = r"C:\Users\gabri\Desktop\FIAP\Fase 4\Aulas\Ligacao_com_banco_de_dados\xlsx_output"

converted = convert_folder(input_folder, output_folder, recurse=False)
display(converted)
# ...existing code...

Nenhum .csv encontrado em: C:\Users\gabri\Desktop\FIAP\Fase 4\Aulas\Ligacao_com_banco_de_dados\data


[]